# Kaggle Submission Notebook (Minimal)

This notebook contains only the code required to reproduce the competition submission CSV:
1. Final retrieval process
2. Retrieval of relevant documents for test queries
3. CSV generation in Kaggle format


In [1]:
from pathlib import Path
import csv
import json
import re

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = Path('/kaggle/input/retrieval-engine-competition')
if not DATA_DIR.exists():
    DATA_DIR = Path('../../data')  # local fallback

OUTPUT_PATH = Path('solutions_SeaFour.csv')
TOP_K = 100
MODEL_NAME = 'bm25'  # 'bm25' or 'tfidf'


In [2]:
def value_to_text(value):
    if value is None:
        return ''
    if isinstance(value, (list, tuple)):
        return ' '.join(str(v) for v in value)
    if pd.isna(value):
        return ''
    return str(value)


def create_content_column(df, columns):
    out = df.copy()
    for col in columns:
        if col not in out.columns:
            out[col] = ''

    merged = []
    for _, row in out[columns].iterrows():
        text = ' '.join(value_to_text(row[col]) for col in columns).strip().lower()
        merged.append(text)

    out['content'] = merged
    out['id'] = out['id'].astype(str)
    return out


token_pattern = re.compile(r'[a-z0-9]+')


def tokenize(text):
    txt = str(text or '').lower()
    txt = re.sub(r'[-_/]', ' ', txt)
    return token_pattern.findall(txt)


In [3]:
def run_tfidf_search(docs_df, queries_df, top_k=100):
    top_k = min(top_k, len(docs_df))

    vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2)
    try:
        doc_vectors = vectorizer.fit_transform(docs_df['content'])
    except ValueError as error:
        if 'After pruning, no terms remain' not in str(error):
            raise
        vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1)
        doc_vectors = vectorizer.fit_transform(docs_df['content'])

    query_vectors = vectorizer.transform(queries_df['content'])

    scores = cosine_similarity(query_vectors, doc_vectors)
    doc_ids = docs_df['id'].to_numpy()

    results = []
    for i, row_scores in enumerate(scores):
        top_idx = np.argsort(row_scores)[-top_k:][::-1]
        results.append({
            'query_id': queries_df.iloc[i]['id'],
            'relevant_docs': doc_ids[top_idx].tolist(),
        })
    return results


def run_bm25_search(docs_df, queries_df, top_k=100):
    try:
        from rank_bm25 import BM25Plus
    except ImportError:
        import subprocess
        import sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'rank-bm25'])
        from rank_bm25 import BM25Plus

    top_k = min(top_k, len(docs_df))

    tokenized_corpus = [tokenize(text) for text in docs_df['content']]
    bm25 = BM25Plus(tokenized_corpus)
    doc_ids = docs_df['id'].to_numpy()

    results = []
    for _, row in queries_df.iterrows():
        query_tokens = tokenize(row['content'])
        scores = bm25.get_scores(query_tokens)
        top_idx = np.argsort(scores)[-top_k:][::-1]
        results.append({
            'query_id': row['id'],
            'relevant_docs': doc_ids[top_idx].tolist(),
        })
    return results


In [4]:
def write_kaggle_submission(results, sample_csv_path, output_csv_path):
    pred_map = {
        str(item['query_id']): [str(doc_id) for doc_id in item['relevant_docs']]
        for item in results
    }

    with open(sample_csv_path, 'r', newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        rows = list(reader)

    if fieldnames is None or len(fieldnames) < 2:
        raise ValueError('Invalid sample submission format.')

    id_col = fieldnames[0]
    pred_col = fieldnames[1]
    category_col = fieldnames[2] if len(fieldnames) >= 3 else None

    with open(output_csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for row in rows:
            qid = str(row[id_col])
            if qid not in pred_map:
                raise ValueError(f'Missing prediction for query_id: {qid}')

            out_row = {
                id_col: qid,
                pred_col: json.dumps(pred_map[qid]),
            }

            if category_col is not None:
                out_row[category_col] = row.get(category_col, '?') or '?'

            writer.writerow(out_row)


In [5]:
docs_df = pd.read_json(DATA_DIR / 'docs.json')
test_queries_df = pd.read_json(DATA_DIR / 'queries_test.json')
sample_submission_path = DATA_DIR / 'submission.csv'

docs_df = create_content_column(docs_df, ['title', 'text', 'tags'])
test_queries_df = create_content_column(test_queries_df, ['title', 'text'])

if MODEL_NAME == 'bm25':
    test_results = run_bm25_search(docs_df, test_queries_df, top_k=TOP_K)
elif MODEL_NAME == 'tfidf':
    test_results = run_tfidf_search(docs_df, test_queries_df, top_k=TOP_K)
else:
    raise ValueError('MODEL_NAME must be "bm25" or "tfidf".')

write_kaggle_submission(test_results, sample_submission_path, OUTPUT_PATH)
print(f'Saved: {OUTPUT_PATH.resolve()}')


Saved: /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project/notebooks/kaggle/solutions_SeaFour.csv


In [6]:
submission_preview = pd.read_csv(OUTPUT_PATH)
submission_preview.head()


,query_id,relevant_doc_ids,category
0,4ffe16bc-5235-418d-9bf3-22d1f2c5796e_145437,"[""6b1a2049-6fc6-429d-a11f-061b10ba3507_96947"",...",?
1,1bb2bb20-7f45-4dcf-a94a-420c454f87b8_56473,"[""da2e5c00-99e4-4e49-9be8-fd34dbe2aba9_120601""...",?
2,6a9a342c-1275-4bb3-a818-8bcce53fac4f_34507,"[""43fa6e5a-6ad6-4701-ba44-68b513409ffc_17053"",...",?
3,cb216e47-add6-41fd-974a-39251e4df3aa_6777,"[""65a10367-e197-471c-8edc-0a73618172e1_14831"",...",?
4,14f1d3f5-8271-400e-9ef2-8319de25c9a1_200748,"[""927af133-bf03-4268-a3bd-94bda5c9da82_118230""...",?
